# MarketScreener — Weekly Stock Screener

**Run:** Weekly, after downloading latest Yahoo Consensus CSV

**Signals:**
1. Estimate Revision Score — TP / Rev / EPS revised UP recently?
2. Price-to-TP Gap — How much upside still unpriced?
3. Price Momentum — Is price already moving in the right direction?
4. Analyst Conviction — Narrow dispersion = higher confidence
5. Combined 1/Rank Score — Same method as your existing notebooks

**Output:**
- Top 10 F&O candidates (20-25 day horizon) — filtered to is_fno=1 universe
- Top 10 Long-term ideas (1yr+) — all tickers with analyst coverage
- Holdings status: VENUSPIPES / ANUP / EMMVEE / ACMESOLAR

In [2]:
# ============================================================
#  CELL 1 - CONFIG
# ============================================================

DB_HOST     = "localhost"
DB_USER     = "towdevuser"
DB_PASSWORD = "Dev703"
DB_NAME     = "timelineofwealth"

EXCEL_PATH   = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\DBInsert\ReportExtract.xlsx"
EXCEL_SHEET  = "AnalystsReco"
YAHOO_FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"
OUTPUT_FOLDER = YAHOO_FOLDER

MIN_ANALYSTS    = 5   # Yahoo minimum analyst count
MIN_ANALYSTS_DB = 2   # broker minimum count
TOP_N           = 10

MY_HOLDINGS = ["VENUSPIPES", "ANUP", "EMMVEE", "ACMESOLAR"]

YAHOO_TO_NSE = {
    # "ANUP.NS": "ANUP",
}

In [3]:
# ============================================================
#  CELL 2 - IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import glob
import os
from datetime import timedelta
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

try:
    import pymysql
    HAS_MYSQL = True
except ImportError:
    print("pymysql not installed. Install with: pip install pymysql")
    HAS_MYSQL = False

def get_conn():
    return pymysql.connect(
        host=DB_HOST, user=DB_USER,
        password=DB_PASSWORD, database=DB_NAME
    )

print("Imports OK")

Imports OK


In [4]:
# ============================================================
#  CELL 3 - LOAD FNO UNIVERSE
#
#  F&O output restricted to is_fno='1' tickers.
#  Long-term list uses all tickers with analyst coverage.
# ============================================================

fno_tickers = set()

if HAS_MYSQL:
    try:
        conn = get_conn()
        df_fno_universe = pd.read_sql(
            "SELECT ticker FROM stock_universe WHERE is_fno = '1'", conn
        )
        conn.close()
        fno_tickers = set(df_fno_universe['ticker'].astype(str).str.strip().unique())
        print(f"FNO universe: {len(fno_tickers)} tickers")
    except Exception as e:
        print(f"Could not load FNO universe: {e}")
        print("F&O filter skipped - all tickers treated as FNO eligible")
else:
    print("No MySQL - FNO filter skipped")

FNO universe: 239 tickers


In [5]:
# ============================================================
#  CELL 4 - LOAD ANALYST RECO  (DB primary, Excel fallback)
# ============================================================

df_db = pd.DataFrame()

if HAS_MYSQL:
    try:
        conn = get_conn()
        df_db = pd.read_sql("""
            SELECT ticker, quarter, date, mcap, cmp,
                   broker, reco, target,
                   sales_y0, sales_y1, sales_y2,
                   ebit_y0,  ebit_y1,  ebit_y2,
                   opm_y0,   opm_y1,   opm_y2,
                   roce_y0,  roce_y1,  roce_y2,
                   valuation_y0, valuation_y1, valuation_y2,
                   analyst_names
            FROM stock_analyst_reco
            WHERE date <= CURDATE()
        """, conn, parse_dates=False)
        conn.close()
        df_db['date'] = pd.to_datetime(df_db['date'], errors='coerce')
        df_db['source'] = 'DB'
        print(f"MySQL: {len(df_db)} rows | quarters: {sorted(df_db['quarter'].unique())}")
    except Exception as e:
        print(f"MySQL failed: {e} - using Excel only")

df_excel_raw = pd.read_excel(EXCEL_PATH, sheet_name=EXCEL_SHEET)
col_map = {
    'Ticker': 'ticker', 'Quarter': 'quarter', 'Date': 'date',
    'Mcap (Cr)': 'mcap', 'Price': 'cmp', 'Broker': 'broker',
    'Reco': 'reco', 'Target': 'target',
    'Rev Y0': 'sales_y0', 'Rev. Y1': 'sales_y1', 'Rev. Y2': 'sales_y2',
    'EBIT Y0': 'ebit_y0', 'EBIT Y1': 'ebit_y1', 'EBIT Y2': 'ebit_y2',
    'EBITDA% Y0': 'opm_y0', 'EBITDA% Y1': 'opm_y1', 'EBITDA% Y2': 'opm_y2',
    'ROCE% Y0': 'roce_y0', 'ROCE% Y1': 'roce_y1', 'ROCE% Y2': 'roce_y2',
    'EV/EBIT Y0': 'valuation_y0', 'EV/EBIT Y1': 'valuation_y1', 'EV/EBIT Y2': 'valuation_y2',
    'Analysts Name': 'analyst_names',
    'EPS Y0': 'eps_y0', 'EPS Y1': 'eps_y1', 'EPS Y2': 'eps_y2',
}
df_excel_raw = df_excel_raw.rename(columns=col_map)
df_excel_raw['date'] = pd.to_datetime(df_excel_raw['date'], errors='coerce')
df_excel_raw['source'] = 'Excel'

if not df_db.empty:
    db_quarters = set(df_db['quarter'].unique())
    df_excel_new = df_excel_raw[~df_excel_raw['quarter'].isin(db_quarters)].copy()
    print(f"   Excel new quarters (not in DB): {sorted(df_excel_new['quarter'].dropna().unique())}")
    df_reco = pd.concat([df_db, df_excel_new], ignore_index=True)
else:
    df_reco = df_excel_raw.copy()
    print("Using Excel as sole source")

for col in ['ticker', 'quarter', 'broker', 'reco']:
    df_reco[col] = df_reco[col].astype(str).str.strip()
for col in ['mcap', 'cmp', 'target', 'sales_y0', 'sales_y1', 'sales_y2', 'ebit_y0', 'ebit_y1', 'ebit_y2']:
    if col in df_reco.columns:
        df_reco[col] = pd.to_numeric(df_reco[col], errors='coerce')

df_reco = df_reco[df_reco['quarter'] != 'nan'].copy()

print(f"\nCombined analyst reco: {len(df_reco)} rows, {df_reco['ticker'].nunique()} tickers")
print(f"Quarters: {sorted(df_reco['quarter'].unique())}")

MySQL: 16012 rows | quarters: ['FY20Q1', 'FY20Q2', 'FY20Q3', 'FY20Q4', 'FY21Q1', 'FY21Q2', 'FY21Q3', 'FY21Q4', 'FY22Q1', 'FY22Q2', 'FY22Q3', 'FY22Q4', 'FY23Q1', 'FY23Q2', 'FY23Q3', 'FY23Q4', 'FY24Q1', 'FY24Q2', 'FY24Q3', 'FY24Q4', 'FY25Q1', 'FY25Q2', 'FY25Q3', 'FY25Q4', 'FY26Q1', 'FY26Q2']
   Excel new quarters (not in DB): ['FY26Q3']

Combined analyst reco: 16648 rows, 535 tickers
Quarters: ['FY20Q1', 'FY20Q2', 'FY20Q3', 'FY20Q4', 'FY21Q1', 'FY21Q2', 'FY21Q3', 'FY21Q4', 'FY22Q1', 'FY22Q2', 'FY22Q3', 'FY22Q4', 'FY23Q1', 'FY23Q2', 'FY23Q3', 'FY23Q4', 'FY24Q1', 'FY24Q2', 'FY24Q3', 'FY24Q4', 'FY25Q1', 'FY25Q2', 'FY25Q3', 'FY25Q4', 'FY26Q1', 'FY26Q2', 'FY26Q3']


In [6]:
# ============================================================
#  CELL 5 - PRICE MOMENTUM  (targeted SQL queries - fast)
#
#  Does NOT load entire nse_price_history table.
#  Instead: finds 6 specific dates, fetches close_price
#  for each date in a small targeted query.
#
#  Window logic:
#    1W = latest - 7  calendar days (find nearest date on/before)
#    2W = latest - 14 calendar days
#    1M = latest - 1  calendar month  (Feb 17 -> Jan 17 or earlier)
#    3M = latest - 3  calendar months
#    6M = latest - 6  calendar months
# ============================================================

df_momentum = pd.DataFrame()

if HAS_MYSQL:
    try:
        conn = get_conn()

        # Step 1: find latest date in DB
        latest_date = pd.to_datetime(
            pd.read_sql("SELECT MAX(date) AS d FROM nse_price_history",
                        conn, parse_dates=False)['d'].iloc[0]
        )
        print(f"Latest price date: {latest_date.date()}")

        # Step 2: compute target calendar dates for each window
        windows = {
            '1W': latest_date - timedelta(days=7),
            '2W': latest_date - timedelta(days=14),
            '1M': latest_date - relativedelta(months=1),
            '3M': latest_date - relativedelta(months=3),
            '6M': latest_date - relativedelta(months=6),
        }

        # Step 3: resolve each window to actual available DB date
        actual_dates = {}
        for label, target_dt in windows.items():
            result = pd.read_sql(
                "SELECT MAX(date) AS d FROM nse_price_history WHERE date <= %s",
                conn, params=(target_dt.strftime('%Y-%m-%d'),), parse_dates=False
            )['d'].iloc[0]
            if result is not None:
                actual_dates[label] = str(result)
                print(f"   {label}: target {target_dt.date()} -> actual {result}")
            else:
                print(f"   {label}: no data available - skipped")

        # Step 4: fetch latest prices
        df_latest_px = pd.read_sql(
            "SELECT nse_ticker AS ticker, close_price AS price_latest "
            "FROM nse_price_history WHERE date = %s",
            conn, params=(latest_date.strftime('%Y-%m-%d'),), parse_dates=False
        )
        df_latest_px['ticker']       = df_latest_px['ticker'].astype(str).str.strip()
        df_latest_px['price_latest'] = pd.to_numeric(df_latest_px['price_latest'], errors='coerce')
        df_latest_px['price_date']   = latest_date.date()

        # Step 5: fetch past prices per window and compute return
        df_momentum = df_latest_px.copy()
        for label, actual_dt_str in actual_dates.items():
            df_past = pd.read_sql(
                "SELECT nse_ticker AS ticker, close_price AS past_price "
                "FROM nse_price_history WHERE date = %s",
                conn, params=(actual_dt_str,), parse_dates=False
            )
            df_past['ticker']     = df_past['ticker'].astype(str).str.strip()
            df_past['past_price'] = pd.to_numeric(df_past['past_price'], errors='coerce')

            df_momentum = df_momentum.merge(df_past, on='ticker', how='left')
            df_momentum[f'ret_{label}'] = (
                (df_momentum['price_latest'] / df_momentum['past_price'] - 1)
                .where(df_momentum['past_price'] > 0)
                .round(4)
            )
            df_momentum.drop(columns=['past_price'], inplace=True)

        conn.close()
        print(f"\nPrice momentum: {len(df_momentum)} tickers as of {latest_date.date()}")

    except Exception as e:
        print(f"Price momentum failed: {e}")
else:
    print("No MySQL - price momentum skipped")

Latest price date: 2026-02-16
   1W: target 2026-02-09 -> actual 2026-02-09
   2W: target 2026-02-02 -> actual 2026-02-02
   1M: target 2026-01-16 -> actual 2026-01-16
   3M: target 2025-11-16 -> actual 2025-11-14
   6M: target 2025-08-16 -> actual 2025-08-14

Price momentum: 2895 tickers as of 2026-02-16


In [7]:
# ============================================================
#  CELL 6 - LOAD YAHOO CONSENSUS CSVs
# ============================================================

yahoo_files = sorted(glob.glob(os.path.join(YAHOO_FOLDER, "*_YahooConsensus.csv")))
df_yahoo = pd.DataFrame()

if not yahoo_files:
    print("No YahooConsensus CSV files found - Yahoo signals skipped")
else:
    dfs = []
    for f in yahoo_files:
        tmp = pd.read_csv(f)
        tmp['Date'] = pd.to_datetime(tmp['Date'])
        dfs.append(tmp)
    df_yahoo = pd.concat(dfs, ignore_index=True)

    if YAHOO_TO_NSE:
        df_yahoo['NSE_Ticker'] = df_yahoo['YAHOO Ticker'].map(YAHOO_TO_NSE).fillna(df_yahoo['YAHOO Ticker'])
    else:
        df_yahoo['NSE_Ticker'] = df_yahoo['YAHOO Ticker']
    df_yahoo['NSE_Ticker'] = df_yahoo['NSE_Ticker'].str.replace(r'\.NS$', '', regex=True)

    for col in ['TP(Avg)', 'TP(Min)', 'TP(Max)', 'Rev.(Avg)', 'EPS(Avg)', 'Current Price']:
        if col in df_yahoo.columns:
            df_yahoo[col] = pd.to_numeric(df_yahoo[col], errors='coerce')
    df_yahoo['#Count'] = pd.to_numeric(df_yahoo.get('#Count', 0), errors='coerce').fillna(0)

    print(f"Yahoo Consensus: {len(yahoo_files)} files")
    print(f"Date range: {df_yahoo['Date'].min().date()} to {df_yahoo['Date'].max().date()}")
    print(f"Tickers: {sorted(df_yahoo['NSE_Ticker'].unique())}")

Yahoo Consensus: 5 files
Date range: 2026-01-09 to 2026-02-13
Tickers: ['360ONE', 'AARTIIND', 'AAVAS', 'ABFRL', 'ABSLAMC', 'ACC', 'ACMESOLAR', 'AFFLE', 'AKZOINDIA', 'ALKYLAMINE', 'AMBER', 'AMBUJACEM', 'ANANDRATHi', 'ANANTRAJ', 'ANGELONE', 'ANUP', 'APLAPOLLO', 'APOLLOHOSP', 'APTUS', 'ARVINDFASN', 'ASHOKLEY', 'ASIANPAINT', 'ASTRAL', 'AUBANK', 'AWL', 'AXISBANK', 'BAJAJ-AUTO', 'BAJAJELEC', 'BAJAJFINSV', 'BAJAJHFL', 'BAJAJHLDNG', 'BAJFINANCE', 'BANDHANBNK', 'BANKBARODA', 'BATAINDIA', 'BAYERCROP', 'BERGEPAINT', 'BHARATFORG', 'BIKAJI', 'BLUESTARCO', 'BORORENEW', 'BOSCHLTD', 'BRIGADE', 'BRITANNIA', 'BSE', 'CAMPUS', 'CAMS', 'CANBK', 'CANFINHOME', 'CARERATING', 'CARTRADE', 'CASTROLIND', 'CCL', 'CDSL', 'CELLO', 'CENTURYPLY', 'CERA', 'CHALET', 'CHOLAFIN', 'CLEAN', 'CMSINFO', 'COFORGE', 'COLPAL', 'CONCOR', 'COROMANDEL', 'CRISIL', 'CUB', 'CYIENT', 'DABUR', 'DALBHARAT', 'DCBBANK', 'DEEPAKNTR', 'DEVYANI', 'DIXON', 'DLF', 'DMART', 'DOMS', 'DREAMFOLKS', 'EASEMYTRIP', 'ECLERX', 'EDELWEISS', 'EICHERMOT', 

In [8]:
# ============================================================
#  CELL 7 - SIGNAL 1: ESTIMATE REVISION SCORE  (Yahoo)
#
#  4-week revision of TP(Min/Avg/Max), Rev Y1, EPS Y1
#  Composite = average of all available revision values
# ============================================================

df_revision_signal = pd.DataFrame()

if not df_yahoo.empty:

    def get_past_date(dates_series, weeks_back):
        target = df_yahoo['Date'].max() - timedelta(weeks=weeks_back)
        eligible = dates_series[dates_series <= target]
        return eligible.max() if not eligible.empty else None

    latest_dt  = df_yahoo['Date'].max()
    past_4w_dt = get_past_date(df_yahoo['Date'].drop_duplicates(), 4)
    print(f"Revision window: {past_4w_dt.date() if past_4w_dt else 'N/A'} to {latest_dt.date()}")

    if past_4w_dt is not None:
        df_L = df_yahoo[df_yahoo['Date'] == latest_dt].copy()
        df_P = df_yahoo[df_yahoo['Date'] == past_4w_dt].copy()
        fy_sorted = sorted(df_yahoo['FIN Yr'].dropna().unique())
        Y1 = fy_sorted[0] if fy_sorted else None

        records = []
        for ticker in df_L['NSE_Ticker'].unique():
            row_L_all = df_L[df_L['NSE_Ticker'] == ticker]
            row_P_all = df_P[df_P['NSE_Ticker'] == ticker]
            if row_L_all.empty or row_P_all.empty:
                continue

            rL, rP = row_L_all.iloc[0], row_P_all.iloc[0]
            count = rL.get('#Count', 0)

            def pct_chg(col):
                l, p = rL.get(col, np.nan), rP.get(col, np.nan)
                if pd.notna(l) and pd.notna(p) and p > 0 and l > 0:
                    return round(l / p - 1, 4)
                return np.nan

            tp_rev_min = pct_chg('TP(Min)')
            tp_rev_avg = pct_chg('TP(Avg)')
            tp_rev_max = pct_chg('TP(Max)')

            rev_rev = np.nan
            eps_rev = np.nan
            if Y1 is not None:
                row_Ly1 = row_L_all[row_L_all['FIN Yr'] == Y1]
                row_Py1 = row_P_all[row_P_all['FIN Yr'] == Y1]
                if not row_Ly1.empty and not row_Py1.empty:
                    rLy, rPy = row_Ly1.iloc[0], row_Py1.iloc[0]
                    if rPy.get('Rev.(Avg)', 0) > 0 and rLy.get('Rev.(Avg)', 0) > 0:
                        rev_rev = round(rLy['Rev.(Avg)'] / rPy['Rev.(Avg)'] - 1, 4)
                    if rPy.get('EPS(Avg)', 0) > 0 and rLy.get('EPS(Avg)', 0) > 0:
                        eps_rev = round(rLy['EPS(Avg)'] / rPy['EPS(Avg)'] - 1, 4)

            vals = [v for v in [tp_rev_min, tp_rev_avg, tp_rev_max, rev_rev, eps_rev] if pd.notna(v)]
            composite = round(np.mean(vals), 4) if vals else np.nan

            records.append({
                'ticker':        ticker,
                'yahoo_count':   count,
                'tp_min':        rL.get('TP(Min)', np.nan),
                'tp_avg':        rL.get('TP(Avg)', np.nan),
                'tp_max':        rL.get('TP(Max)', np.nan),
                'tp_appreciation': rL.get('TP_Appreciation', np.nan),
                'tp_rev_min_4w': tp_rev_min,
                'tp_rev_avg_4w': tp_rev_avg,
                'tp_rev_max_4w': tp_rev_max,
                'rev_rev_4w':    rev_rev,
                'eps_rev_4w':    eps_rev,
                'revision_score': composite,
            })

        df_revision_signal = pd.DataFrame(records)
        df_revision_signal = df_revision_signal[df_revision_signal['yahoo_count'] >= MIN_ANALYSTS].copy()
        print(f"Revision signal: {len(df_revision_signal)} tickers (>= {MIN_ANALYSTS} analysts)")
    else:
        print("Not enough Yahoo history for 4-week window")
else:
    print("Yahoo data not available - revision signal skipped")

Revision window: 2026-01-09 to 2026-02-13
Revision signal: 216 tickers (>= 5 analysts)


In [9]:
# ============================================================
#  CELL 8 - SIGNAL 2: ANALYST RECO SIGNAL  (DB + Excel)
#
#  Latest quarter per ticker. Computes:
#    - pct_buy, tp_median, sales/ebit growth, tp_dispersion
#  NOTE: cmp here = price at report date (stale).
#        upside_pct is recalculated in Cell 9 using
#        price_latest from nse_price_history.
# ============================================================

df_reco_sorted = df_reco[df_reco['quarter'] != 'nan'].copy()
df_reco_sorted = df_reco_sorted.sort_values(['ticker', 'quarter', 'date'], ascending=[True, False, False])
latest_q = df_reco_sorted.groupby('ticker')['quarter'].first().reset_index()
latest_q.columns = ['ticker', 'latest_quarter']

df_latest = df_reco.merge(latest_q, on='ticker')
df_latest = df_latest[
    (df_latest['quarter'] == df_latest['latest_quarter']) &
    (df_latest['quarter'] != 'nan')
].copy()

def reco_agg(group):
    n = len(group)
    buy_count = group['reco'].str.upper().isin(['BUY', 'STRONG BUY', 'ADD', 'ACCUMULATE']).sum()
    pct_buy   = buy_count / n if n > 0 else np.nan
    cmp_at_report = group['cmp'].median()
    tp_median     = group['target'].median()

    s0 = group['sales_y0'].median()
    s1 = group['sales_y1'].median()
    s2 = group['sales_y2'].median()
    sales_g_y1 = (s1/s0 - 1) if (s0 > 0 and s1 > 0) else np.nan
    sales_g_y2 = (s2/s1 - 1) if (s1 > 0 and s2 > 0) else np.nan

    e0 = group['ebit_y0'].median() if 'ebit_y0' in group.columns else np.nan
    e1 = group['ebit_y1'].median() if 'ebit_y1' in group.columns else np.nan
    e2 = group['ebit_y2'].median() if 'ebit_y2' in group.columns else np.nan
    ebit_g_y1 = (e1/e0 - 1) if (pd.notna(e0) and e0>0 and pd.notna(e1) and e1>0) else np.nan
    ebit_g_y2 = (e2/e1 - 1) if (pd.notna(e1) and e1>0 and pd.notna(e2) and e2>0) else np.nan

    tp_cv = (group['target'].std() / group['target'].mean()
             if group['target'].notna().sum() >= 2 else np.nan)

    def r(v): return round(v, 4) if pd.notna(v) else np.nan

    return pd.Series({
        'broker_count':     n,
        'latest_quarter':   group['quarter'].iloc[0],
        'cmp_at_report':    round(cmp_at_report, 2),
        'tp_median':        round(tp_median, 2) if pd.notna(tp_median) else np.nan,
        'pct_buy':          r(pct_buy),
        'sales_g_y1':       r(sales_g_y1),
        'sales_g_y2':       r(sales_g_y2),
        'ebit_g_y1':        r(ebit_g_y1),
        'ebit_g_y2':        r(ebit_g_y2),
        'tp_dispersion_cv': r(tp_cv),
    })

df_reco_agg = df_latest.groupby('ticker').apply(reco_agg).reset_index()
df_reco_agg = df_reco_agg[df_reco_agg['broker_count'] >= MIN_ANALYSTS_DB].copy()

print(f"Analyst reco signal: {len(df_reco_agg)} tickers (>= {MIN_ANALYSTS_DB} brokers)")
print(f"Quarters: {sorted(df_reco_agg['latest_quarter'].unique())}")

Analyst reco signal: 195 tickers (>= 2 brokers)
Quarters: ['FY23Q4', 'FY24Q1', 'FY24Q3', 'FY25Q3', 'FY25Q4', 'FY26Q1', 'FY26Q2', 'FY26Q3']


In [10]:
# ============================================================
#  CELL 9 - COMBINE ALL SIGNALS
#
#  Merge order:
#    1. df_reco_agg  (broker signals - anchor)
#    2. df_momentum  (current price + returns from price history)
#    3. df_revision  (Yahoo enrichment - optional)
#
#  upside_pct = tp_median / price_latest - 1  (current price)
#  cmp_at_report kept for reference only
# ============================================================

df_screen = df_reco_agg.copy()

# Merge current price and momentum returns
if not df_momentum.empty:
    ret_cols = ['ticker', 'price_latest', 'price_date'] + \
               [c for c in df_momentum.columns if c.startswith('ret_')]
    df_screen = df_screen.merge(df_momentum[ret_cols], on='ticker', how='left')
    # Upside using today's price
    df_screen['upside_pct'] = (
        (df_screen['tp_median'] / df_screen['price_latest'] - 1)
        .where(df_screen['price_latest'] > 0)
        .round(4)
    )
else:
    df_screen['price_latest'] = np.nan
    df_screen['price_date']   = None
    # Fallback: use stale report price
    df_screen['upside_pct'] = (
        (df_screen['tp_median'] / df_screen['cmp_at_report'] - 1)
        .where(df_screen['cmp_at_report'] > 0)
        .round(4)
    )

# Merge Yahoo revision
if not df_revision_signal.empty:
    yahoo_cols = ['ticker', 'yahoo_count',
                  'tp_min', 'tp_avg', 'tp_max', 'tp_appreciation',
                  'tp_rev_min_4w', 'tp_rev_avg_4w', 'tp_rev_max_4w',
                  'rev_rev_4w', 'eps_rev_4w', 'revision_score']
    df_screen = df_screen.merge(df_revision_signal[yahoo_cols], on='ticker', how='left')

# Flag thin Yahoo coverage
if 'yahoo_count' in df_screen.columns:
    df_screen['yahoo_thin'] = df_screen['yahoo_count'].fillna(0) <= 2

# Flag FNO eligibility
df_screen['is_fno'] = df_screen['ticker'].isin(fno_tickers) if fno_tickers else True

print(f"Combined universe: {len(df_screen)} tickers")
print(f"FNO eligible : {df_screen['is_fno'].sum()}")
print(f"With Yahoo   : {df_screen['revision_score'].notna().sum() if 'revision_score' in df_screen.columns else 'N/A'}")
print(f"With momentum: {df_screen['ret_1W'].notna().sum() if 'ret_1W' in df_screen.columns else 'N/A'}")

Combined universe: 195 tickers
FNO eligible : 123
With Yahoo   : 118
With momentum: 189


In [11]:
# ============================================================
#  CELL 10 - SCORE & RANK  (1/Rank composite)
# ============================================================

df_s = df_screen.copy()

def rank_signal(series, ascending=False):
    r = series.rank(ascending=ascending, na_option='bottom')
    return 1.0 / r

df_s['rank_revision']  = rank_signal(df_s['revision_score']) if 'revision_score' in df_s.columns else 0.0
df_s['rank_upside']    = rank_signal(df_s['upside_pct'])
df_s['rank_pct_buy']   = rank_signal(df_s['pct_buy'])
df_s['rank_sales_g']   = rank_signal(df_s['sales_g_y1'])
df_s['rank_ebit_g']    = rank_signal(df_s['ebit_g_y1'])
df_s['rank_momentum']  = rank_signal(df_s['ret_1M']) if 'ret_1M' in df_s.columns else 0.0

# F&O Score: short-term momentum + revision heavy
df_s['fno_score'] = (
    3.0 * df_s['rank_revision'] +
    2.0 * df_s['rank_momentum'] +
    1.0 * df_s['rank_upside']   +
    1.0 * df_s['rank_pct_buy']
)

# Long-Term Score: fundamentals + upside heavy
df_s['lt_score'] = (
    3.0 * df_s['rank_upside']   +
    2.0 * df_s['rank_ebit_g']   +
    2.0 * df_s['rank_sales_g']  +
    1.0 * df_s['rank_revision'] +
    1.0 * df_s['rank_pct_buy']
)

print(f"F&O score range: {df_s['fno_score'].min():.3f} to {df_s['fno_score'].max():.3f}")
print(f"LT  score range: {df_s['lt_score'].min():.3f} to {df_s['lt_score'].max():.3f}")

F&O score range: 0.040 to 3.025
LT  score range: 0.049 to 3.035


In [12]:
# ============================================================
#  CELL 11 - TOP 10 F&O CANDIDATES
#  Restricted to is_fno = True
# ============================================================

all_ret_cols = [c for c in df_s.columns if c.startswith('ret_')]

display_cols_fno = [
    'ticker', 'latest_quarter', 'broker_count',
    'price_latest', 'tp_median', 'upside_pct', 'pct_buy',
    'revision_score', 'tp_rev_avg_4w', 'rev_rev_4w', 'eps_rev_4w',
] + all_ret_cols + ['fno_score']
display_cols_fno = [c for c in display_cols_fno if c in df_s.columns]

df_fno_pool = df_s[df_s['is_fno'] == True].copy() if 'is_fno' in df_s.columns else df_s.copy()
df_fno = (
    df_fno_pool
    .sort_values('fno_score', ascending=False)
    .head(TOP_N)[display_cols_fno]
    .reset_index(drop=True)
)
df_fno.index += 1
df_fno['holding'] = df_fno['ticker'].isin(MY_HOLDINGS).map({True: '***', False: ''})

print(f"{'='*70}")
print(f"TOP {TOP_N} F&O CANDIDATES  [{df_fno_pool.shape[0]} FNO stocks screened]")
print(f"{'='*70}")
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 160)
print(df_fno.to_string())

if 'yahoo_thin' in df_s.columns:
    thin = df_fno[df_fno['ticker'].isin(df_s[df_s['yahoo_thin']]['ticker'])]['ticker'].tolist()
    if thin:
        print(f"\nWARN: Thin Yahoo coverage (<= 2 analysts): {thin}")

TOP 10 F&O CANDIDATES  [123 FNO stocks screened]
        ticker latest_quarter  broker_count  price_latest  tp_median  upside_pct  pct_buy  revision_score  tp_rev_avg_4w  rev_rev_4w  eps_rev_4w  ret_1W  ret_2W  ret_1M  ret_3M  ret_6M  fno_score holding
1   MUTHOOTFIN         FY26Q3             2       3498.20     3777.5      0.0798   0.0000          0.1425         0.1222         NaN         NaN -0.0745 -0.0104 -0.1113 -0.0610  0.2687   3.024997        
2     AARTIIND         FY26Q3             2        455.25      525.5      0.1543   1.0000          0.0524         0.0786         NaN         NaN -0.0322  0.2205  0.2838  0.1584  0.2104   2.203852        
3   MANAPPURAM         FY26Q3             2        303.75      312.5      0.0288   0.5000          0.1323         0.0634         NaN         NaN -0.0096  0.0872 -0.0326  0.0804  0.1419   1.530761        
4         MFSL         FY26Q3             2       1848.30     2150.0      0.1632   1.0000          0.1262         0.0816         NaN   

In [13]:
# ============================================================
#  CELL 12 - TOP 10 LONG-TERM IDEAS
#  All tickers (not limited to FNO)
# ============================================================

display_cols_lt = [
    'ticker', 'latest_quarter', 'broker_count',
    'price_latest', 'tp_median', 'upside_pct', 'pct_buy',
    'sales_g_y1', 'sales_g_y2', 'ebit_g_y1', 'ebit_g_y2',
    'revision_score', 'tp_dispersion_cv', 'is_fno', 'lt_score'
]
display_cols_lt = [c for c in display_cols_lt if c in df_s.columns]

df_lt = (
    df_s
    .sort_values('lt_score', ascending=False)
    .head(TOP_N)[display_cols_lt]
    .reset_index(drop=True)
)
df_lt.index += 1
df_lt['holding'] = df_lt['ticker'].isin(MY_HOLDINGS).map({True: '***', False: ''})

print(f"{'='*70}")
print(f"TOP {TOP_N} LONG-TERM IDEAS  [{len(df_s)} total stocks screened]")
print(f"{'='*70}")
print(df_lt.to_string())

TOP 10 LONG-TERM IDEAS  [195 total stocks screened]
        ticker latest_quarter  broker_count  price_latest  tp_median  upside_pct  pct_buy  sales_g_y1  sales_g_y2  ebit_g_y1  ebit_g_y2  revision_score  tp_dispersion_cv  is_fno  lt_score holding
1        NOCIL         FY24Q3             3        146.60      250.0      0.7053   0.0000     -0.0910      0.1405    -0.2405     0.3834             NaN            0.0181   False  3.034590        
2    SIGNATURE         FY26Q3             2       1098.50     1016.5     -0.0746   1.0000      0.0787      0.8868     1.4289     3.5092             NaN            0.0090   False  2.064866        
3      ETERNAL         FY26Q3             2        286.60      287.5      0.0031   0.5000      1.6828      0.7598        NaN        NaN          0.0097            0.3566    True  2.055416        
4   MUTHOOTFIN         FY26Q3             2       3498.20     3777.5      0.0798   0.0000      0.6020      0.1516     0.8914     0.1061          0.1425            0

In [14]:
# ============================================================
#  CELL 13 - HOLDINGS STATUS CHECK
# ============================================================

print(f"{'='*70}")
print("HOLDINGS STATUS")
print(f"{'='*70}")

for h in MY_HOLDINGS:
    row = df_s[df_s['ticker'] == h]
    if row.empty:
        print(f"  {h:15s}  NOT FOUND in screener universe")
        continue
    r = row.iloc[0]

    price   = r.get('price_latest',   np.nan)
    tp      = r.get('tp_median',      np.nan)
    upside  = r.get('upside_pct',     np.nan)
    pct_buy = r.get('pct_buy',        np.nan)
    rev_sc  = r.get('revision_score', np.nan)
    ret_1m  = r.get('ret_1M',         np.nan)
    q       = r.get('latest_quarter', '?')
    brokers = int(r.get('broker_count', 0))

    if (pd.notna(upside) and upside > 0.20 and pd.notna(pct_buy) and pct_buy > 0.70):
        light = 'GREEN'
    elif (pd.notna(upside) and upside > 0.10) or (pd.notna(pct_buy) and pct_buy > 0.50):
        light = 'YELLOW'
    else:
        light = 'RED'

    print(f"""
  [{light}]  {h}  [{q}]  {brokers} brokers
     Current Price  : {price:.1f} -> Median TP: {tp:.1f}
     Upside to TP   : {upside:.1%}
     % BUY reco     : {pct_buy:.0%}
     Revision score : {f'{rev_sc:+.2%}' if pd.notna(rev_sc) else 'N/A (no Yahoo)'}
     1M price return: {f'{ret_1m:+.1%}' if pd.notna(ret_1m) else 'N/A'}
""")

HOLDINGS STATUS
  VENUSPIPES       NOT FOUND in screener universe
  ANUP             NOT FOUND in screener universe
  EMMVEE           NOT FOUND in screener universe
  ACMESOLAR        NOT FOUND in screener universe


In [15]:
# ============================================================
#  CELL 14 - EXPORT TO EXCEL
# ============================================================

from datetime import datetime

run_date = datetime.today().strftime('%Y%m%d')
out_file = os.path.join(OUTPUT_FOLDER, f"{run_date}_MarketScreener.xlsx")

df_export = df_s.copy()
df_export['fno_rank'] = df_export['fno_score'].rank(ascending=False).astype(int)
df_export['lt_rank']  = df_export['lt_score'].rank(ascending=False).astype(int)
df_export = df_export.sort_values('fno_rank')

df_holdings_export = df_s[df_s['ticker'].isin(MY_HOLDINGS)].copy()

with pd.ExcelWriter(out_file, engine='xlsxwriter') as writer:
    df_fno.to_excel(writer,             sheet_name='FNO_Top10',    index=True)
    df_lt.to_excel(writer,              sheet_name='LT_Top10',     index=True)
    df_holdings_export.to_excel(writer, sheet_name='Holdings',     index=False)
    df_export.to_excel(writer,          sheet_name='FullUniverse', index=False)

print(f"Screener saved: {out_file}")

Screener saved: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260217_MarketScreener.xlsx
